# Check AXI yourself

AXI is a layer on the host side, between a model and the state it edits. The model is unchanged. Architecture by **Rajnish Choubey**; tests built and run by Shalendra Gaur. Not affiliated with Google.

Two checks, in order of effort:

1. **A few seconds, no key, no GPU.** Replays the saved real replies of the 4,000-step run (Gemini 3.8 Flash) through this repository's own code and recomputes the held state step by step.
2. **A few minutes, your own Gemini key.** Runs the same harness live for 15 steps and prints the held state, the same replies applied as written, and the prompt size.

Run the cells top to bottom (Runtime > Run all works for check 1; check 2 asks for a key first).

## Check 1: no key

In [ ]:
!git clone --depth 1 https://github.com/gaurshalendra2412/axi-runtime
%cd axi-runtime/python
!python3 quickstart_replay.py

What to look for in the output:

* `held state exactly the right one: 4,000 of 4,000 steps`
* `exactly the ideal reply: 3,954 of 4,000 (98.85%)`
* in the first 400 requests, replies applied as written leave the right state on 85 of 400 steps; the AXI run on 400 of 400; the cost of a step with the right state is 4.5 times higher as written
* the last line says whether every figure matches the published ones

The script stops with an error if one replayed step differs from the saved one.

## Check 2: your own key

Put your key in Colab's **Secrets** (the key icon in the left bar), name it `GEMINI_API_KEY`, and switch on notebook access. A free key from AI Studio is enough: the first 200 steps are answered by the program itself, then Gemini 3.8 Flash is asked for 15 steps. The key is held in memory and is never printed or written to a file.

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
!pip -q install google-genai
!python3 live_check.py

What to look for: held state exactly the right one on every live step; the same replies applied as written leave some steps off the right state; prompt tokens at the first live steps and at the last are about the same, because the host rebuilds the prompt from the held state at every step.

Longer run on a paid key: `!python3 live_check.py --steps 100 --daily-limit 0 --min-interval 0.3`. All three checks are described in `docs/TRY_IT.md`.